# Airline assistant

## Clients and prices

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

azure = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
azure_model = os.getenv("GPT_MODEL", "gpt-4o")

import json
import sqlite3

import gradio as gr

DB = repo_root / "W2-Build Multi Model Chatbots" / "Day-4" / "prices.db"

with sqlite3.connect(DB) as conn:
    conn.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
    )

ticket_prices = {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}


## Price tools

`get_ticket_price` reads SQLite. `set_ticket_price` is the exercise: the assistant can change a fare, not only quote one.

In [2]:
def get_ticket_price(city):
    print(f"get_ticket_price({city})", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.lower(),),
        ).fetchone()
    if not row:
        return f"No price data available for {city}"
    return f"Ticket price to {city} is ${row[0]}"


def set_ticket_price(city, price):
    print(f"set_ticket_price({city}, {price})", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) "
            "ON CONFLICT(city) DO UPDATE SET price = excluded.price",
            (city.lower(), float(price)),
        )
    return f"Updated the ticket price to {city} to ${price}"


for city, price in ticket_prices.items():
    set_ticket_price(city, price)

get_ticket_price("Tokyo")


set_ticket_price(london, 799)
set_ticket_price(paris, 899)
set_ticket_price(tokyo, 1420)
set_ticket_price(sydney, 2999)
get_ticket_price(Tokyo)


'Ticket price to Tokyo is $1420.0'

## Tool schemas


In [ ]:
system_message = """
You are a helpful assistant for an airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Use the tools for ticket prices. Do not invent a price.
"""

city_property = {
    "type": "string",
    "description": "The city the customer wants to travel to",
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the price of a return ticket to the destination city.",
            "parameters": {
                "type": "object",
                "properties": {"destination_city": city_property},
                "required": ["destination_city"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_ticket_price",
            "description": "Set or update the return ticket price for a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": city_property,
                    "price": {
                        "type": "number",
                        "description": "New return fare in USD",
                    },
                },
                "required": ["destination_city", "price"],
                "additionalProperties": False,
            },
        },
    },
]


def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        name = tool_call.function.name
        if name == "get_ticket_price":
            content = get_ticket_price(arguments["destination_city"])
        elif name == "set_ticket_price":
            content = set_ticket_price(
                arguments["destination_city"],
                arguments["price"],
            )
        else:
            content = f"Unknown tool {name}"
        responses.append(
            {
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            }
        )
    return responses


def chat(message, history):
    history = [{"role": turn["role"], "content": turn["content"]} for turn in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = azure.chat.completions.create(
        model=azure_model,
        messages=messages,
        tools=tools,
        timeout=120.0,
    )

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        messages.append(tool_message.model_dump(exclude_none=True))
        messages.extend(handle_tool_calls(tool_message))
        response = azure.chat.completions.create(
            model=azure_model,
            messages=messages,
            tools=tools,
            timeout=120.0,
        )

    return response.choices[0].message.content


gr.ChatInterface(fn=chat, title="FlightAI").launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


get_ticket_price(london)
set_ticket_price(london, 100)


Try `What is the fare to Tokyo?`, then `Set the London fare to 750`, then `Price to London and Tokyo`. The last one should call the price tool twice in one turn.